# Baseline Predictive Maintenance Model

This notebook records the first baseline result for the predictive maintenance classification project,

The goal is to evaluate a Logistic Regression baseline trained on the AI4I 2020 Predictive Maintenance dataset and understand its behavior.

In [1]:
import numpy as np
import pandas as pd

from predictive_maintenance_ml.experiment import BaselineExperiment
from predictive_maintenance_ml.evaluation import ClassificationEvaluator

In [2]:
experiment = BaselineExperiment()
results = experiment.run()

y_test = results["y_test"]
y_pred = results["y_pred"]
y_proba = results["y_proba"]

In [13]:
evaluator = ClassificationEvaluator()

metrics = evaluator.evaluate(
    y_true=y_test,
    y_pred=y_pred,
    y_proba=y_proba,
)

cm = metrics.pop('confusion_matrix')
cl_report = metrics.pop('classification_report')

metrics_df = pd.DataFrame({'Metrics':metrics})

In [18]:
print(cm)
print(cl_report)
metrics_df.head()

[[1593  339]
 [  12   56]]
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



,Metrics
accuracy,0.824500
precision,0.141772
recall,0.823529
f1,0.241901
roc_auc,0.906954


In [12]:
print(metrics)

{'accuracy': 0.8245, 'precision': 0.14177215189873418, 'recall': 0.8235294117647058, 'f1': 0.24190064794816415, 'roc_auc': 0.9069540859822189}


## Baseline result

The Logistic Regression baseline produced the following result on the test set:

- Accuracy: 0.8245
- Precision: 0.1418
- Recall: 0.8235
- F1-score: 0.2419
- ROC-AUC: 0.9070

The confusion matrix was:

|                | Predicted normal | Predicted failure |
|----------------|------------------|-------------------|
| True normal    | 1593             | 339               |
| True failure   | 12               | 56                |

## Interpretation

The model detects most failure cases: it correctly identifies 56 out of 68 failures in the test set. This gives a recall of approximately 82.35% for the failure class.

However, the model also generates many false alarms: 339 normal samples are incorrectly classified as failures. As a result, precision for the failure class is only approximately 14.18%.

This means that the model is aggressive in predicting failures. It is useful if the priority is to avoid missing failures, but it would likely create too many unnecessary inspections in a real maintenance setting.

## Why accuracy is misleading

The dataset is highly imbalanced. In the test set, there are 1932 normal samples and only 68 failure samples.

A naive model that always predicts "normal" would achieve high accuracy, but it would detect no failures. For this reason, accuracy is not the main metric for this problem.

The more relevant metrics are:

- recall for the failure class,
- precision for the failure class,
- F1-score,
- confusion matrix,
- ROC-AUC,
- and later precision-recall analysis.

In [19]:
print(y_test.value_counts(normalize=True))
print(y_test.value_counts())

Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64
Machine failure
0    1932
1      68
Name: count, dtype: int64


## Further developements

The baseline model has a good ROC-AUC score, which suggests that it can rank risky samples reasonably well. However, the default decision threshold produces many false positives.

The next step is to evaluate different probability thresholds and study the tradeoff between precision and recall.

A higher threshold should reduce false positives and improve precision, but it may also increase false negatives and reduce recall.